In [82]:
import numpy as np
import pandas as pd
import os
import gc
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [83]:
def aggregate_table(df: pd.DataFrame, prefix: str, group_var='SK_ID_CURR'):
    df_working = df.copy()

    num_rules = {}

    for col in df.columns:
        if col == group_var:
            continue

        if df[col].dtype in ['int64', 'float64']:
            if 'AMT' in col:
                num_rules[col] = ['sum', 'mean', 'min', 'max']
            elif 'DAYS' in col:
                num_rules[col] = ['min', 'max']
            else:
                num_rules[col] = ['mean', 'max', 'sum']

    cat_cols = df_working.select_dtypes(['object', 'string']).columns.tolist()

    if len(cat_cols) > 0:
        df_cats = pd.get_dummies(
            df_working[[group_var] + cat_cols],
            columns=cat_cols,
            dummy_na=True
        )

        cat_rules = {}

        for col in df_cats.columns:
            if col != group_var:
                cat_rules[col] = ['sum', 'mean']

        cat_agg = df_cats.groupby(group_var).agg(cat_rules)
        cat_agg.columns = [f"{prefix}_{col[0]}_{col[1]}" for col in cat_agg.columns]
    else:
        cat_agg = None

    if len(num_rules) > 0:
        num_cols = [group_var] + list(num_rules.keys())
        df_num = df_working[num_cols]

        num_agg = df_num.groupby(group_var).agg(num_rules)
        num_agg.columns = [f"{prefix}_{col[0]}_{col[1]}" for col in num_agg.columns]
    else:
        num_agg = None

    if (num_agg is not None) and (cat_agg is not None):
        final_agg = num_agg.join(cat_agg, how='outer')
    elif num_agg is not None:
        final_agg = num_agg
    else:
        final_agg = cat_agg

    print(f"[{prefix}] Обработка завершена. Создано признаков: {final_agg.shape[1]}")
    return final_agg


In [84]:
def transform_main_df(df: pd.DataFrame):
    df_processed = df.copy()

    # ИСПРАВЛЕНИЕ 1: В Pandas строковый тип называется 'string', а не 'str'
    cat_cols = df_processed.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    num_cols = df_processed.select_dtypes(include=['number']).columns.tolist()

    special_cols = ['SK_ID_CURR', 'TARGET']
    cat_cols = [col for col in cat_cols if col not in special_cols]
    num_cols = [col for col in num_cols if col not in special_cols]

    # ИСПРАВЛЕНИЕ 2: Вшиваем fillna('missing') прямо в цикл конвертации
    for col in cat_cols:
        df_processed[col] = df_processed[col].fillna('missing').astype(str)

    if 'AMT_CREDIT' in df_processed.columns and 'AMT_INCOME_TOTAL' in df_processed.columns:
        df_processed['MAIN_CREDIT_TO_INCOME_RATIO'] = df_processed['AMT_CREDIT'] / (df_processed['AMT_INCOME_TOTAL'] + 1e-5)

    if 'AMT_ANNUITY' in df_processed.columns and 'AMT_INCOME_TOTAL' in df_processed.columns:
        df_processed['MAIN_ANNUITY_TO_INCOME_RATIO'] = df_processed['AMT_ANNUITY'] / (df_processed['AMT_INCOME_TOTAL'] + 1e-5)

    if 'DAYS_BIRTH' in df_processed.columns:
        df_processed['MAIN_AGE_YEARS'] = -df_processed['DAYS_BIRTH'] / 365.25

    if 'DAYS_EMPLOYED' in df_processed.columns:
        df_processed['MAIN_EMPLOYMENT_YEARS'] = -df_processed['DAYS_EMPLOYED'] / 365.25

    return df_processed

In [85]:
DATA_FOLDER = './data'

# 1. Загружаем обе основные анкеты
train_main = pd.read_csv(os.path.join(DATA_FOLDER, 'application_train.csv'))
test_main = pd.read_csv(os.path.join(DATA_FOLDER, 'application_test.csv'))

# Чтобы не дублировать код для merge, будем использовать список датасетов
datasets = [train_main, test_main]

print("\n--- ЭТАП 1: Балансы бюро ---")
bbal = pd.read_csv(os.path.join(DATA_FOLDER, 'bureau_balance.csv'))
bbal_agg = aggregate_table(bbal, prefix='bbal', group_var='SK_ID_BUREAU')
del bbal; gc.collect()

print("\n--- ЭТАП 2: Кредитное Бюро ---")
bureau = pd.read_csv(os.path.join(DATA_FOLDER, 'bureau.csv'))
bureau = bureau.merge(bbal_agg, on='SK_ID_BUREAU', how='left')
bureau_final = aggregate_table(bureau, prefix='bureau', group_var='SK_ID_CURR')
del bbal_agg, bureau; gc.collect()

for df in datasets:
    df.merge(bureau_final, on='SK_ID_CURR', how='left')
del bureau_final; gc.collect()

print("\n--- ЭТАП 3: Детальные истории (Платежи, POS, Карты) ---")
prev_apps = pd.read_csv(os.path.join(DATA_FOLDER, 'previous_application.csv'))
dual_tables = {'inst': 'installments_payments.csv', 'pos': 'POS_CASH_balance.csv', 'cc': 'credit_card_balance.csv'}

for prefix, file_name in dual_tables.items():
    print(f">> Обработка файла: {file_name}")
    df_raw = pd.read_csv(os.path.join(DATA_FOLDER, file_name))

    # Агрегаты к клиенту
    df_direct = aggregate_table(df_raw, prefix=f"{prefix}_direct", group_var='SK_ID_CURR')
    for df in datasets:
        df.merge(df_direct, on='SK_ID_CURR', how='left')
    del df_direct

    # Агрегаты к прошлому кредиту
    df_micro = aggregate_table(df_raw, prefix=f"{prefix}_micro", group_var='SK_ID_PREV')
    prev_apps = prev_apps.merge(df_micro, on='SK_ID_PREV', how='left')
    del df_micro, df_raw; gc.collect()

print("\n--- ЭТАП 4: Прошлые заявки ---")
prev_final = aggregate_table(prev_apps, prefix='prev', group_var='SK_ID_CURR')
for df in datasets:
    df.merge(prev_final, on='SK_ID_CURR', how='left')
del prev_apps, prev_final; gc.collect()

print("\n--- ЭТАП 5: Зачистка ID ---")
id_cols = ['SK_ID_CURR', 'SK_ID_BUREAU', 'SK_ID_PREV']
for df in datasets:
    df.drop(columns=[c for c in id_cols if c in df.columns], inplace=True)

print("\n--- ЭТАП 6: Бизнес-логика ---")
train_main = transform_main_df(train_main)
test_main = transform_main_df(test_main)

# Теперь у тебя есть два готовых датасета: train_main и test_main
print(f"Размеры: Трейн {train_main.shape}, Тест {test_main.shape}")


--- ЭТАП 1: Балансы бюро ---
[bbal] Обработка завершена. Создано признаков: 21

--- ЭТАП 2: Кредитное Бюро ---
[bureau] Обработка завершена. Создано признаков: 156

--- ЭТАП 3: Детальные истории (Платежи, POS, Карты) ---
>> Обработка файла: installments_payments.csv
[inst_direct] Обработка завершена. Создано признаков: 21
[inst_micro] Обработка завершена. Создано признаков: 21
>> Обработка файла: POS_CASH_balance.csv
[pos_direct] Обработка завершена. Создано признаков: 38
[pos_micro] Обработка завершена. Создано признаков: 38
>> Обработка файла: credit_card_balance.csv
[cc_direct] Обработка завершена. Создано признаков: 91
[cc_micro] Обработка завершена. Создано признаков: 91

--- ЭТАП 4: Прошлые заявки ---
[prev] Обработка завершена. Создано признаков: 879

--- ЭТАП 5: Зачистка ID ---

--- ЭТАП 6: Бизнес-логика ---
Размеры: Трейн (307511, 125), Тест (48744, 124)


In [86]:
drop_cols = ['SK_ID_CURR', 'TARGET']

features = [col for col in train_main.columns if col not in drop_cols]

y = train_main['TARGET']
x = train_main[features]

x_test = test_main

# x_test = x_test.reindex(columns=x.columns)

# missing_in_test = x_test.isna().sum().sum()
# print(f"После выравнивания в тесте найдено {missing_in_test} пропущенных ячеек.")

In [87]:
cat_features = x.select_dtypes(include=['object', 'string']).columns.tolist()

for col in cat_features:
    x[col] = x[col].fillna('missing').astype(str)

    # Защищаем и тестовую выборку тоже
    if col in x_test.columns:
        x_test[col] = x_test[col].fillna('missing').astype(str)

print(f"Обучаем модель на {len(features)} признаках. Найдено категорий: {len(cat_features)}")

n_folds = 5
folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(x))
test_predictions = np.zeros(len(x_test))


for fold, (train_idx, val_idx) in enumerate(folds.split(x, y)):
    print(f"\n=== СТАРТ ОБУЧЕНИЯ ФОЛДА № {fold + 1} ===")

    # Режем данные на обучающую выборку (4/5) и валидационную (1/5)
    x_train, y_train = x.iloc[train_idx], y.iloc[train_idx]
    x_val, y_val = x.iloc[val_idx], y.iloc[val_idx]

    # Оборачиваем данные в специальный сверхбыстрый формат CatBoost Pool
    train_pool = Pool(x_train, y_train, cat_features=cat_features)
    val_pool = Pool(x_val, y_val, cat_features=cat_features)
    test_pool = Pool(x_test, cat_features=cat_features)

    # Инициализируем саму модель CatBoost со строгими настройками
    model = CatBoostClassifier(
        iterations=3000,              # Максимум 3000 деревьев
        learning_rate=0.03,           # Скорость обучения (чтобы модель училась плавно)
        depth=6,                      # Оптимальная глубина деревьев для этой задачи
        eval_metric='AUC',            # Наша главная целевая метрика
        random_seed=42,
        task_type='CPU',              # Если есть мощная видеокарта, можно заменить на 'GPU'
        verbose=200                   # Выводить лог обучения каждые 200 деревьев
    )

    # Обучаем модель. Передаем val_pool для контроля переобучения
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=150,    # Тот самый ручной тормоз (остановка, если 150 шагов нет роста)
        use_best_model=True           # В конце зафиксировать веса лучшей итерации
    )

    # Делаем предсказание вероятностей (нам нужен именно класс 1 - дефолт, поэтому [:, 1])
    val_preds = model.predict_proba(val_pool)[:, 1]
    oof_predictions[val_idx] = val_preds

    # Считаем ROC-AUC для текущего фолда
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"--> Успех! Скор фолда {fold + 1} ROC-AUC: {fold_auc:.5f}")

    # ИНТЕГРАЦИЯ: Делаем предсказание на тест текущей моделью и добавляем долю (1/5) в общий котел
    test_predictions += model.predict_proba(test_pool)[:, 1] / n_folds


    del x_train, x_val, train_pool, val_pool, model
    gc.collect()

overall_auc = roc_auc_score(y, oof_predictions)
print(f"\n==========================================")
print(f" ОБЩИЙ КРОСС-ВАЛИДАЦИОННЫЙ ROC-AUC: {overall_auc:.5f}")
print(f"==========================================")

Обучаем модель на 124 признаках. Найдено категорий: 16

=== СТАРТ ОБУЧЕНИЯ ФОЛДА № 1 ===
0:	test: 0.6686799	best: 0.6686799 (0)	total: 138ms	remaining: 6m 53s
200:	test: 0.7460710	best: 0.7460710 (200)	total: 25s	remaining: 5m 47s
400:	test: 0.7517434	best: 0.7517434 (400)	total: 50.2s	remaining: 5m 25s
600:	test: 0.7548030	best: 0.7548030 (600)	total: 1m 17s	remaining: 5m 8s
800:	test: 0.7558611	best: 0.7558618 (797)	total: 1m 51s	remaining: 5m 5s
1000:	test: 0.7565827	best: 0.7565851 (999)	total: 2m 18s	remaining: 4m 36s
1200:	test: 0.7570144	best: 0.7570144 (1200)	total: 2m 46s	remaining: 4m 8s
1400:	test: 0.7573505	best: 0.7573555 (1396)	total: 3m 14s	remaining: 3m 41s
1600:	test: 0.7576907	best: 0.7576907 (1600)	total: 3m 41s	remaining: 3m 13s
1800:	test: 0.7578815	best: 0.7578819 (1789)	total: 4m 9s	remaining: 2m 45s
2000:	test: 0.7581200	best: 0.7581654 (1986)	total: 4m 36s	remaining: 2m 18s
2200:	test: 0.7582093	best: 0.7582794 (2137)	total: 5m 5s	remaining: 1m 50s
2400:	test: 

KeyError: 'SK_ID_CURR'